## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-17-12-53-27 +0000,nyt,Ukraine Attacks Kerch Bridge Linking Russia to...,https://www.nytimes.com/2026/11/17/world/europ...
1,2026-11-17-12-52-00 +0000,wsj,Deutsche Bank Shares Drop After Leaks Put Spot...,https://www.wsj.com/finance/deutsche-bank-shar...
2,2026-11-17-12-44-53 +0000,bbc,Speed camera fault may have allowed drivers to...,https://www.bbc.co.uk/news/articles/c5y4j7z373...
3,2026-11-17-12-44-00 +0000,wsj,The Morning Risk Report: Commerce Department M...,https://www.wsj.com/risk-compliance-journal/th...
4,2026-11-17-12-43-07 +0000,bbc,"Government must act now or lose Royal Mail, fo...",https://www.bbc.co.uk/news/articles/cjvg26p3z2...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2754/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
95,trump,57
140,nvidia,29
457,china,22
225,will,20
367,ai,17


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
367,2026-11-16-17-02-28 +0000,chitri,President Trump backs campaign to reinstate Nv...,https://www.chicagotribune.com/2026/11/16/nvid...,139
410,2026-11-16-14-46-32 +0000,nypost,Trump ‘would not mind’ Jensen Huang reinstated...,https://nypost.com/2026/11/16/us-news/trump-wo...,136
347,2026-11-16-17-48-36 +0000,nyt,Trump Says He’s Open to Reinstatement of Jense...,https://www.nytimes.com/2026/11/16/us/politics...,131
350,2026-11-16-17-43-00 +0000,wsj,Nvidia Scandal Threatens China’s Charm Offensi...,https://www.wsj.com/world/china/nvidia-scandal...,129
389,2026-11-16-15-50-59 +0000,nypost,Trump defends Commerce boss Lutnick over Nvidi...,https://nypost.com/2026/11/16/us-news/trump-de...,128


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
367,139,2026-11-16-17-02-28 +0000,chitri,President Trump backs campaign to reinstate Nv...,https://www.chicagotribune.com/2026/11/16/nvid...
96,93,2026-11-17-10-22-09 +0000,bbc,How new US chip restrictions on South East Asi...,https://www.bbc.co.uk/news/articles/cwq6e2qgvm...
61,68,2026-11-17-11-05-00 +0000,wsj,Nvidia formed a board committee to oversee nat...,https://www.wsj.com/politics/national-security...
101,59,2026-11-17-10-09-28 +0000,nypost,Russia says Ukraine struck its bridge to Crime...,https://nypost.com/2026/11/17/world-news/russi...
340,56,2026-11-16-18-09-00 +0000,wsj,The Fed is modeling how AI could transform job...,https://www.wsj.com/economy/central-banking/wa...
194,54,2026-11-17-01-00-00 +0000,wsj,"Fresh off big state-level victories, Democrats...",https://www.wsj.com/politics/policy/in-blue-wa...
317,52,2026-11-16-19-30-30 +0000,bbc,Lindsay Clancy to face retrial next year for d...,https://www.bbc.co.uk/news/articles/cw5y1v8de3...
420,48,2026-11-16-13-43-51 +0000,cbc,Bank of Canada likely to hold interest rates s...,https://www.cbc.ca/news/business/cpi-inflation...
100,42,2026-11-17-10-11-07 +0000,wsj,Stock Market News: Tech Futures Seesaw Ahead o...,https://www.wsj.com/livecoverage/nvidia-earnin...
345,39,2026-11-16-18-01-25 +0000,nypost,Passengers enduring 60-hour travel delays stil...,https://nypost.com/2026/11/16/business/passeng...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
